# EndoGSLAM with Innovations — 10-Scene Evaluation (post-fix)

Runs the full pipeline with all 5 bug fixes applied.

**Pipeline:**
1. Setup environment + patched CUDA rasterizer
2. Sanity-check the rasterizer (4-output return, valid `gauss_vis`)
3. Download C3VD dataset
4. **Parity test**: run sigmoid_t3_a with innovations OFF and baseline iters (15+25) → should match the EndoGSLAM baseline (~22.24 PSNR, 0.37 ATE)
5. Run our method (innovations ON, 30+50 iters) on all 10 scenes
6. Compute metrics and compare against baseline

**Expected baseline numbers:**
| scene | PSNR | ATE |
|---|---|---|
| cecum_t1_b | 20.85 | 0.92 |
| cecum_t2_b | 17.53 | 0.58 |
| cecum_t3_a | 22.82 | 0.28 |
| sigmoid_t1_a | 25.53 | 0.24 |
| sigmoid_t2_a | 19.81 | 0.14 |
| sigmoid_t3_a | 22.24 | 0.37 |
| trans_t1_b | 25.25 | 0.30 |
| trans_t2_c | 20.83 | 0.17 |
| trans_t4_a | 18.16 | 0.75 |
| trans_t4_b | 22.53 | 0.24 |
| **AVG** | **21.56** | **0.42** |

## 1. Environment Setup

In [ ]:
# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Clone (or pull if already exists)
import os
if not os.path.exists('/content/project'):
    !git clone https://github.com/baimingyang98/Endoscopic-3DGS-SLAM-with-Visibility-Pruning.git /content/project
else:
    !cd /content/project && git pull
%cd /content/project

# Show the latest commit so we know which version we're running
!git log -3 --format='  %h  %s'

In [ ]:
# Install dependencies
!pip install -q tqdm numpy Pillow opencv-python imageio matplotlib kornia natsort pyyaml plotly lpips open3d torchmetrics pytorch-msssim trimesh pandas

## 2. Patch and Install Modified CUDA Rasterizer

In [ ]:
# Clone base rasterizer (only if not already there)
import os
if not os.path.exists('/content/rasterizer'):
    !git clone https://github.com/JonathonLuiten/diff-gaussian-rasterization-w-depth.git /content/rasterizer

In [ ]:
# Apply visibility patches
%cd /content/project
!python patch_rasterizer.py /content/rasterizer

In [ ]:
# Build and install
!pip install /content/rasterizer/

In [ ]:
# Sanity check the rasterizer (4-output return, valid gauss_vis, etc.)
%cd /content/project
!python scripts/check_rasterizer.py

## 3. Download C3VD Dataset

In [ ]:
%cd /content/project
!pip install -q gdown
import os
if not os.path.exists('/content/project/data/C3VD/sigmoid_t3_a'):
    !mkdir -p data/C3VD
    !gdown 1MwpfFKKweM1L3bYY7V4dYzS_IlkFaOUe -O /content/C3VD_EndoGSLAM.tar.gz
    !tar -xzf /content/C3VD_EndoGSLAM.tar.gz -C data/
else:
    print('Data already extracted, skipping download.')
!ls data/C3VD/

## 4. Parity Test (innovations OFF, baseline iters)

This cell runs `sigmoid_t3_a` with **all innovations disabled** and the original 15+25 iteration counts.

**Expected:** PSNR ≈ 22.24, ATE ≈ 0.37 (matching the EndoGSLAM baseline within ±0.5).

**If this matches baseline → all 5 fixes are working.** 
**If this doesn't match → there's still a bug; do not proceed with the 10-scene run.**

In [ ]:
%%writefile /content/project/configs/c3vd/c3vd_parity.py
"""Parity test config: all innovations OFF, baseline iters.
Should reproduce the original EndoGSLAM-H paper numbers within noise.
"""
import os

scenes = [
    "cecum_t1_b", "cecum_t2_b", "cecum_t3_a",
    "sigmoid_t1_a", "sigmoid_t2_a", "sigmoid_t3_a",
    "trans_t1_b", "trans_t2_c", "trans_t4_a", "trans_t4_b",
]

primary_device = "cuda:0"
seed = 0
try:
    scene_name = scenes[int(os.environ["SCENE_NUM"])]
except (KeyError, IndexError):
    scene_name = "sigmoid_t3_a"

map_every = 1
keyframe_every = 8
tracking_iters = 15
mapping_iters = 25

group_name = "C3VD_parity"
run_name = scene_name

config = dict(
    workdir=f"./experiments/{group_name}",
    run_name=run_name,
    seed=seed,
    primary_device=primary_device,
    map_every=map_every,
    keyframe_every=keyframe_every,
    distance_keyframe_selection=True,
    distance_current_frame_prob=0.1,
    mapping_window_size=-1,
    report_global_progress_every=999999,
    scene_radius_depth_ratio=3,
    mean_sq_dist_method="projective",
    report_iter_progress=False,
    load_checkpoint=False,
    checkpoint_time_idx=0,
    save_checkpoints=False,
    checkpoint_interval=int(1e10),
    gaussian_simplification=True,
    data=dict(
        basedir="./data/C3VD",
        gradslam_data_cfg="./configs/data/c3vd.yaml",
        sequence=scene_name,
        desired_image_height=1080 // 2,
        desired_image_width=1350 // 2,
        start=0, end=-1, stride=1, num_frames=-1,
        train_or_test="train",
    ),
    tracking=dict(
        use_gt_poses=False, forward_prop=True,
        num_iters=tracking_iters,
        use_sil_for_loss=True, sil_thres=0.99,
        use_l1=True, ignore_outlier_depth_loss=False,
        loss_weights=dict(im=0.5, depth=1.0),
        lrs=dict(
            means3D=0.0, rgb_colors=0.0, unnorm_rotations=0.0,
            logit_opacities=0.0, log_scales=0.0,
            cam_unnorm_rots=0.002, cam_trans=0.005,
        ),
    ),
    mapping=dict(
        num_iters=mapping_iters,
        add_new_gaussians=True, sil_thres=0.5,
        use_l1=True, use_sil_for_loss=False,
        ignore_outlier_depth_loss=False,
        loss_weights=dict(im=1.0, depth=1.0),
        lrs=dict(
            means3D=0.0001, rgb_colors=0.0025, unnorm_rotations=0.001,
            logit_opacities=0.05, log_scales=0.001,
            cam_unnorm_rots=0.0, cam_trans=0.0,
        ),
        prune_gaussians=True,
        pruning_dict=dict(
            start_after=0, remove_big_after=0, stop_after=20,
            prune_every=20, removal_opacity_threshold=0.005,
            final_removal_opacity_threshold=0.005,
            reset_opacities=False, reset_opacities_every=int(1e10),
        ),
        use_gaussian_splatting_densification=False,
        densify_dict=dict(
            start_after=500, remove_big_after=3000, stop_after=5000,
            densify_every=100, grad_thresh=0.0002, num_to_split_into=2,
            removal_opacity_threshold=0.005, final_removal_opacity_threshold=0.005,
            reset_opacities_every=3000,
        ),
    ),
    # ALL innovations OFF for parity
    innovations=dict(
        enable_visibility_pruning=False,
        enable_periodic_ba=False,
        enable_deformation=False,
    ),
    viz=dict(
        render_mode="color", offset_first_viz_cam=True,
        show_sil=False, visualize_cams=False,
        viz_w=320, viz_h=320, viz_near=0.01, viz_far=100.0,
        view_scale=2, viz_fps=30,
        enter_interactive_post_online=True,
    ),
)

In [ ]:
import subprocess, time, os
%cd /content/project

env = os.environ.copy()
env['SCENE_NUM'] = '5'  # sigmoid_t3_a

os.makedirs('experiments/C3VD_parity/sigmoid_t3_a', exist_ok=True)
log_file = 'experiments/C3VD_parity/sigmoid_t3_a/run.log'

print('Running parity test on sigmoid_t3_a (innovations OFF, 15+25 iters)...')
start = time.time()
with open(log_file, 'w') as log:
    proc = subprocess.run(
        ['python', 'scripts/main.py', 'configs/c3vd/c3vd_parity.py'],
        env=env, stdout=log, stderr=subprocess.STDOUT,
        cwd='/content/project'
    )
elapsed = (time.time() - start) / 60
print(f"  {'OK' if proc.returncode==0 else 'FAILED'} in {elapsed:.1f} min")
if proc.returncode != 0:
    with open(log_file) as f:
        for line in f.readlines()[-25:]:
            print('  ' + line.rstrip())

In [ ]:
# Compute parity metrics
%cd /content/project
!python scripts/calc_metrics.py --experiment_dir ./experiments/C3VD_parity/sigmoid_t3_a --gt_dir ./data/C3VD/sigmoid_t3_a

### ⚠️ STOP and check the parity result above

- **PSNR ≈ 22.0–22.5** AND **ATE ≤ 0.5** → ✅ Fixes work; proceed to section 5.
- **PSNR < 21.5** OR **ATE > 0.7** → ❌ Bug remains; do not run sections 5-7.

## 5. Write Best Config (innovations ON, 30+50 iters)

In [ ]:
%%writefile /content/project/configs/c3vd/c3vd_best.py
"""Best configuration: BA + Visibility Pruning (eta=0.90) with all bug fixes."""
import os

scenes = [
    "cecum_t1_b", "cecum_t2_b", "cecum_t3_a",
    "sigmoid_t1_a", "sigmoid_t2_a", "sigmoid_t3_a",
    "trans_t1_b", "trans_t2_c", "trans_t4_a", "trans_t4_b",
]

primary_device = "cuda:0"
seed = 0
try:
    scene_name = scenes[int(os.environ["SCENE_NUM"])]
except (KeyError, IndexError):
    scene_name = "sigmoid_t3_a"

map_every = 1
keyframe_every = 8
tracking_iters = 30
mapping_iters = 50

group_name = "C3VD_best"
run_name = scene_name

config = dict(
    workdir=f"./experiments/{group_name}",
    run_name=run_name,
    seed=seed,
    primary_device=primary_device,
    map_every=map_every,
    keyframe_every=keyframe_every,
    distance_keyframe_selection=True,
    distance_current_frame_prob=0.1,
    mapping_window_size=-1,
    report_global_progress_every=999999,
    scene_radius_depth_ratio=3,
    mean_sq_dist_method="projective",
    report_iter_progress=False,
    load_checkpoint=False,
    checkpoint_time_idx=0,
    save_checkpoints=False,
    checkpoint_interval=int(1e10),
    gaussian_simplification=True,
    data=dict(
        basedir="./data/C3VD",
        gradslam_data_cfg="./configs/data/c3vd.yaml",
        sequence=scene_name,
        desired_image_height=1080 // 2,
        desired_image_width=1350 // 2,
        start=0, end=-1, stride=1, num_frames=-1,
        train_or_test="train",
    ),
    tracking=dict(
        use_gt_poses=False, forward_prop=True,
        num_iters=tracking_iters,
        use_sil_for_loss=True, sil_thres=0.99,
        use_l1=True, ignore_outlier_depth_loss=False,
        loss_weights=dict(im=0.5, depth=1.0),
        lrs=dict(
            means3D=0.0, rgb_colors=0.0, unnorm_rotations=0.0,
            logit_opacities=0.0, log_scales=0.0,
            cam_unnorm_rots=0.002, cam_trans=0.005,
        ),
    ),
    mapping=dict(
        num_iters=mapping_iters,
        add_new_gaussians=True, sil_thres=0.5,
        use_l1=True, use_sil_for_loss=False,
        ignore_outlier_depth_loss=False,
        loss_weights=dict(im=1.0, depth=1.0),
        lrs=dict(
            means3D=0.0001, rgb_colors=0.0025, unnorm_rotations=0.001,
            logit_opacities=0.05, log_scales=0.001,
            cam_unnorm_rots=0.0, cam_trans=0.0,
        ),
        prune_gaussians=True,
        pruning_dict=dict(
            start_after=0, remove_big_after=0, stop_after=20,
            prune_every=20, removal_opacity_threshold=0.005,
            final_removal_opacity_threshold=0.005,
            reset_opacities=False, reset_opacities_every=int(1e10),
        ),
        use_gaussian_splatting_densification=False,
        densify_dict=dict(
            start_after=500, remove_big_after=3000, stop_after=5000,
            densify_every=100, grad_thresh=0.0002, num_to_split_into=2,
            removal_opacity_threshold=0.005, final_removal_opacity_threshold=0.005,
            reset_opacities_every=3000,
        ),
    ),
    innovations=dict(
        enable_visibility_pruning=True,
        distance_gamma=0.5,
        degeneration_eta=0.9,
        vis_threshold=0.05,
        min_observations=50,
        vis_window_size=15,
        enable_periodic_ba=True,
        ba_every_m_frames=50,
        ba_n_keyframes=5,
        ba_num_iters=20,
        ba_selection="hybrid",
        ba_lrs=dict(
            means3D=0.00002, rgb_colors=0.0005, unnorm_rotations=0.0002,
            logit_opacities=0.01, log_scales=0.0002,
            cam_unnorm_rots=0.0005, cam_trans=0.001,
        ),
        # Innovation 3 disabled (C3VD is rigid)
        enable_deformation=False,
        deform_lr=0.0005,
        var_threshold=0.1,
        lambda_deform_temporal=0.1,
        lambda_deform_magnitude=0.01,
        enable_deform_weighted_tracking=False,
    ),
    viz=dict(
        render_mode="color", offset_first_viz_cam=True,
        show_sil=False, visualize_cams=False,
        viz_w=320, viz_h=320, viz_near=0.01, viz_far=100.0,
        view_scale=2, viz_fps=30,
        enter_interactive_post_online=True,
    ),
)

## 6. Run All 10 Scenes (innovations ON)

In [ ]:
import subprocess, time, os
%cd /content/project

scenes = [
    "cecum_t1_b",    # 0
    "cecum_t2_b",    # 1
    "cecum_t3_a",    # 2
    "sigmoid_t1_a",  # 3
    "sigmoid_t2_a",  # 4
    "sigmoid_t3_a",  # 5
    "trans_t1_b",    # 6
    "trans_t2_c",    # 7
    "trans_t4_a",    # 8
    "trans_t4_b",    # 9
]

results = {}
batch_start = time.time()

for idx, scene in enumerate(scenes):
    print(f"\n{'='*60}")
    print(f"Scene {idx+1}/10: {scene}")
    print(f"{'='*60}")

    start_time = time.time()
    env = os.environ.copy()
    env['SCENE_NUM'] = str(idx)

    os.makedirs(f'experiments/C3VD_best/{scene}', exist_ok=True)
    log_file = f'experiments/C3VD_best/{scene}/run.log'

    with open(log_file, 'w') as log:
        proc = subprocess.run(
            ['python', 'scripts/main.py', 'configs/c3vd/c3vd_best.py'],
            env=env, stdout=log, stderr=subprocess.STDOUT,
            cwd='/content/project',
        )

    elapsed = time.time() - start_time
    status = 'OK' if proc.returncode == 0 else f'FAILED (code {proc.returncode})'
    results[scene] = {'status': status, 'time_min': elapsed / 60}

    print(f"  Status: {status}")
    print(f"  Time: {elapsed/60:.1f} min")

    if proc.returncode != 0:
        with open(log_file, 'r') as f:
            lines = f.readlines()
        print('  Last 20 lines of log:')
        for line in lines[-20:]:
            print('    ' + line.rstrip())

total_min = (time.time() - batch_start) / 60
print(f"\n{'='*60}")
print(f'BATCH COMPLETE  -  total: {total_min:.1f} min ({total_min/60:.1f} h)')
print(f"{'='*60}")
for scene, info in results.items():
    print(f"  {scene:20s} | {info['status']:12s} | {info['time_min']:.1f} min")

## 7. Compute Metrics for All Scenes

In [ ]:
%cd /content/project
!python scripts/calc_metrics.py --all --group_dir ./experiments/C3VD_best --data_dir ./data/C3VD

In [ ]:
# Display results table side-by-side with baseline
import pandas as pd
import os

csv_path = '/content/project/experiments/C3VD_best/metrics_summary.csv'
if not os.path.exists(csv_path):
    print('No metrics CSV found.')
else:
    df = pd.read_csv(csv_path)

    baseline = {
        'cecum_t1_b':   (20.85, 0.92),
        'cecum_t2_b':   (17.53, 0.58),
        'cecum_t3_a':   (22.82, 0.28),
        'sigmoid_t1_a': (25.53, 0.24),
        'sigmoid_t2_a': (19.81, 0.14),
        'sigmoid_t3_a': (22.24, 0.37),
        'trans_t1_b':   (25.25, 0.30),
        'trans_t2_c':   (20.83, 0.17),
        'trans_t4_a':   (18.16, 0.75),
        'trans_t4_b':   (22.53, 0.24),
    }

    print(f"{'scene':<14} {'PSNR':>7} {'(base)':>7} {'ΔPSNR':>7}   {'ATE':>7} {'(base)':>7} {'ΔATE':>7}")
    print('-' * 70)
    psnrs, ates = [], []
    for _, row in df.iterrows():
        s = row['scene']
        b_psnr, b_ate = baseline.get(s, (None, None))
        d_psnr = row['PSNR'] - b_psnr if b_psnr is not None else 0
        d_ate = row['ATE'] - b_ate if b_ate is not None else 0
        print(f"{s:<14} {row['PSNR']:>7.2f} {b_psnr:>7.2f} {d_psnr:>+7.2f}   "
              f"{row['ATE']:>7.3f} {b_ate:>7.3f} {d_ate:>+7.3f}")
        psnrs.append(row['PSNR']); ates.append(row['ATE'])

    base_psnr_avg = sum(b for b, _ in baseline.values()) / len(baseline)
    base_ate_avg = sum(a for _, a in baseline.values()) / len(baseline)
    print('-' * 70)
    print(f"{'AVERAGE':<14} {sum(psnrs)/len(psnrs):>7.2f} {base_psnr_avg:>7.2f} "
          f"{sum(psnrs)/len(psnrs) - base_psnr_avg:>+7.2f}   "
          f"{sum(ates)/len(ates):>7.3f} {base_ate_avg:>7.3f} "
          f"{sum(ates)/len(ates) - base_ate_avg:>+7.3f}")

## 8. Render Comparison Videos

**Two ways to get the 'Baseline' panel in the comparison video:**

### Option A (default, free) — use the parity run as Baseline

Section 4 already produced `experiments/C3VD_parity/sigmoid_t3_a/`. That run uses *our* code with all innovations OFF and the original 15+25 iters. After the bug fixes, the parity numbers should match the EndoGSLAM baseline within ±0.5 PSNR — so this is **functionally equivalent** to the original baseline. Cell 25 below uses this.

### Option B (literal baseline) — run the original EndoGSLAM (Loping151) and use its output

Cells 26–27 clone the original EndoGSLAM repo, run it on `sigmoid_t3_a` (~1.5h on T4), and produce a 3-way comparison using the literal original code. Use this if your tutor wants to see the *exact* original baseline.

### Option A — render using the parity run

In [ ]:
# Render comparison video using the parity test as the Baseline panel
%cd /content/project
!mkdir -p /content/videos
!python scripts/render_video.py \
    --gt_dir          ./data/C3VD/sigmoid_t3_a \
    --experiment_dir  ./experiments/C3VD_parity/sigmoid_t3_a \
    --experiment_dir  ./experiments/C3VD_best/sigmoid_t3_a \
    --label           Baseline \
    --label           Ours \
    --include_depth \
    --fps             20 \
    --output          /content/videos/sigmoid_t3_a_compare.mp4

### Option B — clone & run the original EndoGSLAM, then render with literal baseline

This adds ~1.5 hours but gives a video using the truly original baseline code. Run cells 26 and 27 in sequence (skip if Option A is enough).

In [ ]:
# Clone the original EndoGSLAM (Loping151) and run it on sigmoid_t3_a.
# Output goes to /content/EndoGSLAM/experiments/C3VD_base/sigmoid_t3_a/.
import os, subprocess, time

if not os.path.exists('/content/EndoGSLAM'):
    !git clone https://github.com/Loping151/EndoGSLAM.git /content/EndoGSLAM
%cd /content/EndoGSLAM

# Symlink the dataset so we don't re-download
!mkdir -p data
if not os.path.exists('/content/EndoGSLAM/data/C3VD'):
    !ln -sfn /content/project/data/C3VD /content/EndoGSLAM/data/C3VD

# Patch the original config to suppress progress (matches our parity setup)
import re
cfg_path = '/content/EndoGSLAM/configs/c3vd/c3vd_base.py'
cfg = open(cfg_path).read()
cfg = re.sub(r'report_iter_progress\s*=\s*\w+', 'report_iter_progress = False', cfg, count=1)
cfg = re.sub(r'report_global_progress_every\s*=\s*\d+', 'report_global_progress_every = 999999', cfg, count=1)
open(cfg_path, 'w').write(cfg)

# IMPORTANT: original EndoGSLAM uses the STOCK rasterizer (3-output return),
# but we already installed the PATCHED one (4-output). The patched one is
# backward-compatible for color rendering -- the only difference is an
# extra returned tensor that the original code ignores via its 3-tuple
# unpack. We need to monkey-patch the original code to accept 4 outputs.

files_to_patch = [
    '/content/EndoGSLAM/scripts/main.py',
    '/content/EndoGSLAM/utils/eval_helpers.py',
]
for fp in files_to_patch:
    if not os.path.exists(fp):
        continue
    src = open(fp).read()
    # color render: 'im, radius, _ = ' -> 'im, radius, _, _ = '
    src = re.sub(
        r'(\w+,\s*\w+,\s*_)\s*=\s*Renderer\(',
        r'\1, _ = Renderer(',
        src,
    )
    # depth_sil render: 'depth_sil, _, _ = ' -> 'depth_sil, _, _, _ = '
    src = re.sub(
        r'(\w+,\s*_,\s*_)\s*=\s*Renderer\(',
        r'\1, _ = Renderer(',
        src,
    )
    open(fp, 'w').write(src)
    print(f'Patched 4-output renderer unpack in {fp}')

# Run on sigmoid_t3_a
env = os.environ.copy()
env['SCENE_NUM'] = '5'  # sigmoid_t3_a
log_file = '/content/orig_baseline.log'
print(f'\nRunning original EndoGSLAM on sigmoid_t3_a...')
start = time.time()
with open(log_file, 'w') as log:
    proc = subprocess.run(
        ['python', 'scripts/main.py', 'configs/c3vd/c3vd_base.py'],
        env=env, stdout=log, stderr=subprocess.STDOUT,
        cwd='/content/EndoGSLAM',
    )
elapsed = (time.time() - start) / 60
print(f"  {'OK' if proc.returncode==0 else 'FAILED'} in {elapsed:.1f} min")
if proc.returncode != 0:
    with open(log_file) as f:
        for line in f.readlines()[-25:]:
            print('  ' + line.rstrip())

In [ ]:
# Render 3-way comparison using the LITERAL original baseline
%cd /content/project
!mkdir -p /content/videos
!python scripts/render_video.py \
    --gt_dir          ./data/C3VD/sigmoid_t3_a \
    --experiment_dir  /content/EndoGSLAM/experiments/C3VD_base/sigmoid_t3_a \
    --experiment_dir  ./experiments/C3VD_best/sigmoid_t3_a \
    --label           'Baseline (original)' \
    --label           'Ours' \
    --include_depth \
    --fps             20 \
    --output          /content/videos/sigmoid_t3_a_orig_compare.mp4

### Optional: batch-render all 10 scenes (uses parity as baseline)

In [ ]:
# Batch-render all 10 scenes (skips any without completed experiments)
%cd /content/project
!python scripts/render_video.py --batch_c3vd \
    --gt_root         ./data/C3VD \
    --baseline_group  ./experiments/C3VD_parity \
    --ours_group      ./experiments/C3VD_best \
    --include_depth \
    --fps             20 \
    --output_dir      /content/videos

In [ ]:
# Preview a video inline in the notebook
from IPython.display import Video, display
import os
# Try Option B first (literal original baseline), fall back to Option A
candidates = [
    '/content/videos/sigmoid_t3_a_orig_compare.mp4',
    '/content/videos/sigmoid_t3_a_compare.mp4',
]
shown = False
for vid in candidates:
    if os.path.exists(vid):
        print(f'Showing: {vid}')
        display(Video(vid, embed=True, width=900))
        shown = True
        break
if not shown:
    print('No video found yet — run cell 25 (Option A) or cells 26+27 (Option B) first.')

## 9. Download Results

In [ ]:
%cd /content/project
# Package experiments + parity + videos for download
!tar -czf /content/results.tar.gz \
    experiments/C3VD_best/metrics_summary.csv \
    experiments/C3VD_best/*/eval/est_w2c.txt \
    experiments/C3VD_best/*/eval/gt_train_w2c.txt \
    experiments/C3VD_best/*/runtimes.txt \
    experiments/C3VD_best/*/run.log \
    experiments/C3VD_parity/sigmoid_t3_a/eval/ \
    experiments/C3VD_parity/sigmoid_t3_a/run.log 2>/dev/null
print('Saved: /content/results.tar.gz')
import os
if os.path.exists('/content/results.tar.gz'):
    print(f'  Size: {os.path.getsize("/content/results.tar.gz")/1e6:.1f} MB')

# Videos as a separate archive (can be large)
if os.path.exists('/content/videos'):
    !tar -czf /content/videos.tar.gz -C /content videos
    if os.path.exists('/content/videos.tar.gz'):
        print(f'Saved: /content/videos.tar.gz')
        print(f'  Size: {os.path.getsize("/content/videos.tar.gz")/1e6:.1f} MB')